# Ch 16. Positional Encoding — Solutions

> This notebook provides reproducible solutions to all five exercises. Outputs are cleared, and code cells run top-to-bottom in a CPU-only environment.


## Problem 1

**Problem**: Calculate and compare the frequencies of dimensions 0 and 32 in sinusoidal PE.

### Solution

For even dimension $2i$, angular frequency is $\omega_i=10000^{-2i/d}$. With $d=64$, dimension 0 has $i=0$ and frequency 1, while dimension 32 has $i=16$ and frequency $10^{-2}=0.01$; it varies 100 times more slowly with position.


In [ ]:
import numpy as np

d_model = 64
dimensions = np.array([0, 32])
frequencies = 10000.0 ** (-dimensions / d_model)
periods = 2 * np.pi / frequencies
assert np.allclose(frequencies, [1.0, 0.01])
print({int(d): {"angular_frequency": float(w), "period": round(float(p), 3)}
       for d, w, p in zip(dimensions, frequencies, periods)})


## Problem 2

**Problem**: Train learnable PE for 100 positions and visualize the difference between positions 0–50 and 51–99.

### Solution

Learned PE does not automatically guarantee smoothness across positions. Choose a reproducible small objective, train 100 vectors, then compare the two ranges by mean, variance, and adjacent differences rather than relying only on visual impressions.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Learn 100 position vectors against a fixed smooth target and compare the requested halves.
rng = np.random.default_rng(1602)
position = np.arange(100)[:, None]
frequency = np.arange(1, 9)[None, :] / 20
target = np.sin(position * frequency)
pe = rng.normal(scale=0.5, size=target.shape)
losses = []
for _ in range(120):
    error = pe - target
    losses.append(float(np.mean(error**2)))
    pe -= 0.2 * error
half_gap = float(np.linalg.norm(pe[:50].mean(0) - pe[50:].mean(0)))
adjacent = [float(np.linalg.norm(np.diff(block, axis=0), axis=1).mean()) for block in (pe[:51], pe[50:])]
fig, axis = plt.subplots(figsize=(6, 2.5)); axis.imshow(pe.T, aspect="auto", cmap="coolwarm"); plt.close(fig)
assert losses[-1] < 1e-12 and half_gap > 0
print({"initial_loss": round(losses[0], 5), "final_loss": losses[-1],
       "half_mean_gap": round(half_gap, 4), "adjacent_change": np.round(adjacent, 4).tolist()})


## Problem 3

**Problem**: Apply ALiBi’s per-head slope formula $m_h = 2^{-8h/H}$ and simulate its effect on long sequences.

### Solution

The slope $m_h$ decreases exponentially with head index. Because the bias at distance $r$ is $-m_hr$, large-slope heads strongly prefer nearby tokens while small-slope heads preserve longer-range context.


In [ ]:
import numpy as np

heads, length = 8, 128
h = np.arange(1, heads + 1)
slopes = 2.0 ** (-8 * h / heads)
distance = np.arange(length)
bias = -slopes[:, None] * distance[None, :]
weights = np.exp(bias - bias.max(axis=1, keepdims=True)); weights /= weights.sum(axis=1, keepdims=True)
expected_distance = weights @ distance
assert np.all(np.diff(slopes) < 0) and np.all(np.diff(expected_distance) > 0)
print({"slopes": slopes.round(6).tolist(), "expected_attended_distance": expected_distance.round(2).tolist()})


## Problem 4

**Problem**: Verify RoPE’s “depends only on relative distance” property for various position pairs.

### Solution

For a 2-D rotation block, $(R_mq)^T(R_nk)=q^TR_{n-m}k$. Thus shifting both positions by the same amount leaves the inner product unchanged. The check below confirms equal inner products for three pairs with the same relative distance.


In [ ]:
import numpy as np

rng = np.random.default_rng(1604)
q, k = rng.normal(size=(2, 2))
theta = 0.17
def rotate(vector, position):
    angle = theta * position
    matrix = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    return matrix @ vector
pairs = [(2, 9), (11, 18), (40, 47)]
scores = np.array([rotate(q, m) @ rotate(k, n) for m, n in pairs])
reference = q @ rotate(k, pairs[0][1] - pairs[0][0])
assert np.allclose(scores, reference, atol=1e-12)
print({"relative_distance": 7, "pair_scores": scores.round(10).tolist(), "max_error": float(np.max(np.abs(scores-reference)))})


## Problem 5

**Problem**: Apply Position Interpolation, $m \to m \cdot L_{train}/L_{test}$, to RoPE and explain its extrapolation effect.

### Solution

Multiplying positions by $s=L_{train}/L_{test}<1$ compresses the maximum test-time rotation angle into the training range. For twice the length, $s=1/2$. This sacrifices positional resolution but avoids unseen excessive phase rotations.


In [ ]:
import numpy as np

train_length, test_length = 2048, 8192
scale = train_length / test_length
positions = np.array([0, test_length // 2, test_length - 1])
raw_angles = positions.astype(float)
interpolated_angles = positions * scale
assert interpolated_angles.max() < train_length and raw_angles.max() >= train_length
print({"scale": scale, "positions": positions.tolist(), "raw_angles": raw_angles.tolist(),
       "interpolated_angles": interpolated_angles.tolist(), "inside_training_range": True})
